<a href="https://colab.research.google.com/github/Git-Divyanshi-shukla/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Git-Divyanshi-shukla/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/Git-Divyanshi-shukla/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("CTR range:", round(df["ctr"].min(), 4), "to", round(df["ctr"].max(), 4))

Rows: 30000
Columns: 44
CTR range: 0.0 to 100.0


I would frame this as a classification task. The goal is to classify content items based on whether they achieve higher or lower CTR. This fits classification because the outcome can be represented as a category, while content characteristics and placement factors can be used as input features. The result can provide directional decision-support for content teams.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
median_ctr = df["ctr"].median()

df["high_ctr"] = (df["ctr"] >= median_ctr).astype(int)

print("Median CTR:", round(median_ctr, 4))
print("\nTarget counts:")
print(df["high_ctr"].value_counts())
print("\nTarget proportions:")
print(df["high_ctr"].value_counts(normalize=True).round(3))

Median CTR: 0.07

Target counts:
high_ctr
1    15190
0    14810
Name: count, dtype: int64

Target proportions:
high_ctr
1    0.506
0    0.494
Name: proportion, dtype: float64


The target will be a binary CTR outcome created from the observed CTR values. I will use the median CTR as a simple threshold: content with CTR at or above the median will be treated as higher-CTR, and content below the median will be treated as lower-CTR. This is a defined proxy based on an observed outcome, rather than a directly observed business label. It should therefore be treated as a directional decision-support target.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [7]:
from sklearn.metrics import precision_score

print("Target balance:")
print(df["high_ctr"].value_counts(normalize=True).round(3))

# A simple baseline: predict every item as the higher-CTR class
baseline_predictions = [1] * len(df)

baseline_precision = precision_score(
    df["high_ctr"],
    baseline_predictions
)

print("Baseline Precision:", round(baseline_precision, 3))

Target balance:
high_ctr
1    0.506
0    0.494
Name: proportion, dtype: float64
Baseline Precision: 0.506


I will use Precision as the main success metric. Precision measures how often the items predicted as higher-CTR are actually in the higher-CTR group. This is useful because the practical goal is to identify content that appears promising without treating every prediction as a guarantee. A higher precision means the selected items are more often aligned with the observed higher-CTR outcome.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
lane_df = df[
    [
        "content_type",
        "position_tier",
        "ctr",
        "impressions_90d",
        "high_ctr"
    ]
].copy()

print("Shape:", lane_df.shape)
print("\nMissing values:")
print(lane_df.isna().sum())

lane_df.head()

Shape: (30000, 5)

Missing values:
content_type       0
position_tier      0
ctr                0
impressions_90d    0
high_ctr           0
dtype: int64


,content_type,position_tier,ctr,impressions_90d,high_ctr
0,keyword article,striking,0.76,3803,1
1,keyword article,page_3_5,0.05,15320,0
2,keyword article,page_3_5,0.09,12581,1
3,keyword article,page_1,0.49,11751,1
4,keyword article,page_3_5,0.13,19140,1


The unit of analysis is one content item or content observation represented by one row in the dataset. Each row contains characteristics such as content type and position tier, along with observed CTR and impressions over 90 days. For this task, the model will use the content-level row as the basic unit for analysis.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [11]:
print("Average CTR by content type:")
print(
    df.groupby("content_type")["ctr"]
      .mean()
      .sort_values(ascending=False)
      .round(4)
)

print("\nAverage CTR by position tier:")
print(
    df.groupby("position_tier")["ctr"]
      .mean()
      .sort_values(ascending=False)
      .round(4)
)

Average CTR by content type:
content_type
feedly article        2.7913
keyword article       0.3448
comparison article    0.1312
Name: ctr, dtype: float64

Average CTR by position tier:
position_tier
top_3       1.4836
page_1      0.6525
striking    0.3232
page_3_5    0.2225
deep        0.1502
Name: ctr, dtype: float64


A fixed rule may be too simple because CTR can vary across different content types and placement positions, and these factors may interact with each other. Instead of using one simple if-statement, ML can learn patterns from multiple features and evaluate whether those patterns help identify higher-CTR observations. The result should still be treated as directional decision-support rather than a guarantee of future performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.